# Semantic Scholar API — Module Overview

This notebook demonstrates the `semantic_scholar.py` module against real Fuster dataset records.
It covers:

1. **Data loading** — select 5 representative Fuster records (Dryad, Zenodo, Semantic Scholar sources)
2. **`get_paper_by_doi()`** — primary lookup path via DOI
3. **`get_paper_by_title()`** — title-search fallback
4. **`get_paper_citations()`** — papers that cite the article
5. **`get_paper_references()`** — papers cited by the article
6. **Caching & rate-limiting** — joblib cache behavior and polite 1 req/sec throttle
7. **Summary table** — consolidated results per record

**Reference:** `src/llm_metadata/semantic_scholar.py` | Plan task SS-6.5

## 1. Setup

In [ ]:
import os
import time
import pprint
import pandas as pd

import sys
sys.path.insert(0, '../src')

import llm_metadata.semantic_scholar as ss

print('BASE_URL:', ss.BASE_URL)
print('API key present:', bool(ss.SEMANTIC_SCHOLAR_API_KEY))

### Environment configuration

The module reads two optional env vars at import time:

| Var | Purpose | Default |
|-----|---------|--------|
| `SEMANTIC_SCHOLAR_API_BASE` | Base URL (override to use proxy) | `https://api.semanticscholar.org/graph/v1` |
| `SEMANTIC_SCHOLAR_API_KEY` | Auth header for dedicated rate limit | *(empty → shared public pool)* |

In the devcontainer, `SEMANTIC_SCHOLAR_API_BASE` points to the nginx reverse proxy that injects the API key — the app code never sees the secret.  
Locally, omitting both vars hits the public endpoint directly with no authentication.

## 2. Load Fuster Records

In [ ]:
df = pd.read_excel('../data/dataset_092624_validated.xlsx')

print(f'Total records: {len(df)}')
print()
print('Records by source:')
print(df['source'].value_counts().to_string())

In [ ]:
# 5 representative records: 1 Dryad, 1 Zenodo, 3 Semantic Scholar
# Selected for clean ASCII titles and journal DOIs
SAMPLE_DOIS = [
    '10.1371/journal.pone.0128238',    # Dryad — wildlife habitat selection
    '10.1093/jhered/esx103',           # Zenodo — wood turtle mating patterns
    '10.1086/671900',                  # SS — redhorse swimming physiology
    '10.3389/fmars.2021.637546',       # SS — benthic coastal communities
    '10.1007/s10661-013-3497-4',       # SS — electrofishing vs snorkeling
]

def _normalise_doi(raw):
    """Strip URL prefix from DOI field."""
    if not isinstance(raw, str):
        return raw
    for prefix in ('https://doi.org/', 'http://doi.org/', 'doi:'):
        if raw.lower().startswith(prefix):
            return raw[len(prefix):]
    return raw

# Build a tidy lookup table from the validated dataset
df['doi_clean'] = df['cited_article_doi'].apply(_normalise_doi)
samples = df[df['doi_clean'].isin(SAMPLE_DOIS)][['source', 'title', 'doi_clean', 'is_oa']].copy()
samples = samples.drop_duplicates('doi_clean').set_index('doi_clean').reindex(SAMPLE_DOIS)
samples

## 3. `get_paper_by_doi()` — Primary Lookup

In [ ]:
# Retrieve metadata for all 5 sample DOIs
papers_by_doi = {}
for doi in SAMPLE_DOIS:
    paper = ss.get_paper_by_doi(doi)
    papers_by_doi[doi] = paper
    status = 'FOUND' if paper else 'NOT FOUND'
    title_snippet = (paper.get('title', '')[:60] + '...') if paper else '—'
    print(f'[{status}] {doi}\n         {title_snippet}\n')

### Raw API response structure

The function requests the fields declared in `DEFAULT_PAPER_FIELDS`:
```
paperId, title, abstract, year, authors, openAccessPdf, externalIds
```

In [ ]:
# Inspect the raw response for one record
example_doi = '10.1371/journal.pone.0128238'
example_paper = papers_by_doi[example_doi]

if example_paper:
    print(f'Keys returned: {list(example_paper.keys())}\n')

    print(f'paperId:   {example_paper.get("paperId")}')
    print(f'title:     {example_paper.get("title")}')
    print(f'year:      {example_paper.get("year")}')
    print(f'externalIds: {example_paper.get("externalIds")}')
    
    oa_pdf = example_paper.get('openAccessPdf')
    print(f'openAccessPdf: {oa_pdf}')
    
    authors = example_paper.get('authors', [])
    print(f'authors ({len(authors)}): {[a.get("name") for a in authors[:3]]}...')

## 4. `get_paper_by_title()` — Title Search Fallback

Used when a DOI lookup returns `None` (e.g., preprints not indexed by DOI, or malformed DOI strings).  
The function issues a title search and returns the first hit.

In [ ]:
# Simulate a fallback: try a title that might have a bad DOI
fallback_title = 'Comparative Physiology and Relative Swimming Performance of Three Redhorse'

result = ss.get_paper_by_title(fallback_title)
if result:
    print(f'Found via title search:')
    print(f'  paperId: {result.get("paperId")}')
    print(f'  title:   {result.get("title")}')
    print(f'  year:    {result.get("year")}')
else:
    print('Not found via title search')

In [ ]:
# Fields returned by title search differ from DOI lookup:
# DEFAULT_SEARCH_FIELDS = "paperId, title, abstract, year, authors" (no openAccessPdf)
if result:
    print(f'Keys from title search: {list(result.keys())}')
    print(f'(note: openAccessPdf not included — use get_paper_by_doi for that field)')

## 5. `get_paper_citations()` — Papers That Cite the Article

In [ ]:
# Use the SS paper ID (not DOI) from the get_paper_by_doi result
example_paper_id = papers_by_doi[example_doi]['paperId'] if papers_by_doi[example_doi] else None
print(f'paperId for citations lookup: {example_paper_id}')

if example_paper_id:
    citations = ss.get_paper_citations(example_paper_id, limit=10)
    print(f'\nFound {len(citations)} citing papers (limit=10):')
    for c in citations[:5]:
        print(f'  - {c.get("title", "(no title)")[:70]}')

In [ ]:
# Citation records are wrapped under 'citingPaper' key in the raw API response;
# the module unwraps them so the returned list is flat paper dicts.
if citations:
    print(f'Citation record keys: {list(citations[0].keys())}')

## 6. `get_paper_references()` — Papers Cited by the Article

In [ ]:
if example_paper_id:
    refs = ss.get_paper_references(example_paper_id, limit=10)
    print(f'Found {len(refs)} references (limit=10):')
    for r in refs[:5]:
        print(f'  - {r.get("title", "(no title)")[:70]}')

In [ ]:
# References are wrapped under 'citedPaper' key in the raw API; module unwraps them.
if refs:
    print(f'Reference record keys: {list(refs[0].keys())}')

## 7. Caching — joblib `Memory`

All four functions are decorated with `@memory.cache` (joblib), storing results in `./cache/`.
A second call with identical arguments returns the cached result instantly — no network request.

In [ ]:
# First call (or cache hit if already cached)
t0 = time.perf_counter()
_ = ss.get_paper_by_doi('10.1371/journal.pone.0128238')
t1 = time.perf_counter()

# Second call — always a cache hit
t2 = time.perf_counter()
_ = ss.get_paper_by_doi('10.1371/journal.pone.0128238')
t3 = time.perf_counter()

print(f'Call 1: {(t1-t0)*1000:.1f} ms  (network or cache)')
print(f'Call 2: {(t3-t2)*1000:.1f} ms  (cache hit)')
print()
print('Cache directory:', ss.memory.location)

## 8. Rate Limiting

The internal `_get()` helper enforces **1 request/second** when an API key is present  
(introductory authenticated limit). Without a key it relies on the shared public pool.  
On HTTP 429 it backs off exponentially: 2 s → 4 s → 8 s (up to 3 retries).

In [ ]:
# Inspect rate-limiting configuration from the module
print(f'Authenticated RPS limit : {ss._AUTHENTICATED_RPS} req/sec')
print(f'Retry delays on 429     : {ss._RETRY_DELAYS} seconds')
print(f'Request timeout         : {ss.REQUEST_TIMEOUT} seconds')
print()
print('Note: caching means repeated notebook runs do not re-hit the API,'
      ' so the rate limiter only fires on cold-cache calls.')

## 9. Summary Table

For each of the 5 sample Fuster records: paper found?, citation count, reference count, OA PDF URL.

In [ ]:
rows = []
for doi in SAMPLE_DOIS:
    src_info = samples.loc[doi] if doi in samples.index else {}
    paper = papers_by_doi.get(doi)

    if paper:
        pid = paper['paperId']
        cites = ss.get_paper_citations(pid, limit=100)
        refs  = ss.get_paper_references(pid, limit=100)
        oa_pdf = (paper.get('openAccessPdf') or {}).get('url')
    else:
        pid = None
        cites, refs, oa_pdf = [], [], None

    rows.append({
        'source'       : src_info.get('source', '?'),
        'doi'          : doi,
        'title'        : (src_info.get('title') or '')[:55] + '…',
        'paper_found'  : paper is not None,
        'citation_count': len(cites),
        'reference_count': len(refs),
        'oa_pdf_url'   : oa_pdf or '—',
    })

summary = pd.DataFrame(rows)
summary

In [ ]:
# Quick stats
found = summary['paper_found'].sum()
with_oa = (summary['oa_pdf_url'] != '—').sum()
print(f'Papers found   : {found}/{len(summary)} ({100*found/len(summary):.0f}%)')
print(f'With OA PDF    : {with_oa}/{len(summary)} ({100*with_oa/len(summary):.0f}%)')
print(f'Avg citations  : {summary[summary["paper_found"]]["citation_count"].mean():.1f}')
print(f'Avg references : {summary[summary["paper_found"]]["reference_count"].mean():.1f}')

## Key Takeaways

| Aspect | Detail |
|--------|---------|
| **Primary lookup** | `get_paper_by_doi()` — returns `None` cleanly on 404 |
| **Fallback** | `get_paper_by_title()` when DOI is missing/malformed |
| **Citation graph** | `get_paper_citations()` + `get_paper_references()` for network analysis |
| **OA PDF** | `openAccessPdf.url` field in DOI response; `get_open_access_pdf_url()` helper |
| **Caching** | joblib `Memory('./cache')` — reproducible runs, no redundant API calls |
| **Rate limiting** | 1 req/sec (authenticated), exponential backoff on 429 |
| **Config** | `SEMANTIC_SCHOLAR_API_BASE` + `SEMANTIC_SCHOLAR_API_KEY` env vars; safe defaults |

The module is a drop-in Stage 1 data-ingestion component: connect to it the same way as `openalex.py` or `dryad.py`.